# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [26]:
# Write your code below.
%load_ext dotenv
%dotenv

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [27]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [31]:
import os
from glob import glob

# Write your code below.
# Load the enviroment variable
PRICE_DATA = os.getenv("PRICE_DATA")

# Find all parquet files in the directory
parquet_files = glob(os.path.join(PRICE_DATA,"**/*.parquet"), recursive=True)

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [56]:
# Write your code below.
import pandas as pd

dd_px = dd.read_parquet(parquet_files).set_index("ticker")

dd_shift = (
    dd_px
        .groupby('ticker', group_keys=False)
        .apply(
            lambda x: x.sort_values('Date', ascending = True)
                       .assign(Close_lag_1 = x['Close'].shift(1),
                               Adj_Close_lag_1 = x['Adj Close'].shift(1)), 
            meta = pd.DataFrame(data ={'Date': 'datetime64[ns]',
                    'Open': 'f8',
                    'High': 'f8',
                    'Low': 'f8',
                    'Close': 'f8',
                    'Adj Close': 'f8',
                    'Volume': 'i8',
                    'source': 'object',
                    'Year': 'int32',
                    'Close_lag_1': 'f8',
                    'Adj_Close_lag_1': 'f8'},
                    index = pd.Index([], dtype=pd.StringDtype(), name='ticker'))
    ))


dd_feat = dd_shift.assign(
    Returns = lambda x: x['Close']/x['Close_lag_1'] - 1,
    hi_lo_range = lambda x: x['High'] - x['Low']
)

dd_feat.compute()

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,Returns,hi_lo_range
ticker,,,,,,,,,,,,,
ACA,2018-11-05,28.389999,30.070000,28.070000,29.129999,28.874557,1963500.0,ACA.csv,2018,NaN,NaN,NaN,2.000000
ACA,2018-11-06,29.209999,29.500000,28.330000,28.330000,28.081572,1883400.0,ACA.csv,2018,29.129999,28.874557,-0.027463,1.170000
ACA,2018-11-07,28.450001,30.360001,27.570000,29.490000,29.231401,1640200.0,ACA.csv,2018,28.330000,28.081572,0.040946,2.790001
ACA,2018-11-08,29.500000,34.915001,29.280001,32.930000,32.641239,2718900.0,ACA.csv,2018,29.490000,29.231401,0.116650,5.635000
ACA,2018-11-09,32.889999,33.514000,31.549999,32.580002,32.294304,1394200.0,ACA.csv,2018,32.930000,32.641239,-0.010629,1.964001
...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZIXI,2020-03-26,4.060000,4.530000,3.880000,4.510000,4.510000,1668500.0,ZIXI.csv,2020,8.660000,8.660000,-0.479215,0.650000
ZIXI,2020-03-27,4.490000,4.710000,4.100000,4.600000,4.600000,1146800.0,ZIXI.csv,2020,8.030000,8.030000,-0.427148,0.610000
ZIXI,2020-03-30,4.830000,4.870000,4.440000,4.640000,4.640000,1212000.0,ZIXI.csv,2020,7.650000,7.650000,-0.393464,0.430000


,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,Returns,hi_lo_range
ticker,,,,,,,,,,,,,
ACA,2018-11-05,28.389999,30.070000,28.070000,29.129999,28.874557,1963500.0,ACA.csv,2018,NaN,NaN,NaN,2.000000
ACA,2018-11-06,29.209999,29.500000,28.330000,28.330000,28.081572,1883400.0,ACA.csv,2018,29.129999,28.874557,-0.027463,1.170000
ACA,2018-11-07,28.450001,30.360001,27.570000,29.490000,29.231401,1640200.0,ACA.csv,2018,28.330000,28.081572,0.040946,2.790001
ACA,2018-11-08,29.500000,34.915001,29.280001,32.930000,32.641239,2718900.0,ACA.csv,2018,29.490000,29.231401,0.116650,5.635000
ACA,2018-11-09,32.889999,33.514000,31.549999,32.580002,32.294304,1394200.0,ACA.csv,2018,32.930000,32.641239,-0.010629,1.964001
...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZIXI,2020-03-26,4.060000,4.530000,3.880000,4.510000,4.510000,1668500.0,ZIXI.csv,2020,8.660000,8.660000,-0.479215,0.650000
ZIXI,2020-03-27,4.490000,4.710000,4.100000,4.600000,4.600000,1146800.0,ZIXI.csv,2020,8.030000,8.030000,-0.427148,0.610000
ZIXI,2020-03-30,4.830000,4.870000,4.440000,4.640000,4.640000,1212000.0,ZIXI.csv,2020,7.650000,7.650000,-0.393464,0.430000


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [25]:
# Write your code below.



Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.